# 🏦 EconoApp — Plataforma de Gestión Económica
## Corte 2 · Arquitectura Profesional con POO + SQLite

---

| Campo | Detalle |
|---|---|
| **Reto** | Economía |
| **Versión** | 2.0 — Escalada |
| **Tecnologías** | Python · POO · SQLite · CRUD |

### Arquitectura del Sistema
```
┌─────────────────────────────────────────────┐
│              CAPA DE PRESENTACIÓN            │
│          Menú interactivo (while)            │
├─────────────────────────────────────────────┤
│              CAPA DE NEGOCIO (POO)           │
│  Persona → Cliente / Analista               │
│  Activo  → Inversion / CuentaBancaria       │
│  Transaccion → Ingreso / Egreso             │
├─────────────────────────────────────────────┤
│           CAPA DE PERSISTENCIA              │
│    SQLite · 3 tablas · PK · FK · JOIN       │
└─────────────────────────────────────────────┘
```

---
## Celda 1 · Importaciones y Configuración Global

In [ ]:
import sqlite3
import os
from datetime import datetime

# ── Nombre del archivo de base de datos ──────────────────────────────────────
DB_NAME = "base_datos_startup_grupo_1.db"

print("✅ Librerías cargadas correctamente.")
print(f"📂 Base de datos: {DB_NAME}")

---
## Celda 2 · Jerarquía de Clases (POO)

### Jerarquía completa:
```
Persona  (padre)
  ├── Cliente   (hijo)
  └── Analista  (hijo)

Activo   (padre)
  ├── Inversion      (hijo)
  └── CuentaBancaria (hijo)

Transaccion (padre)
  ├── Ingreso (hijo)
  └── Egreso  (hijo)
```

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  JERARQUÍA 1 · Persona → Cliente / Analista
# ══════════════════════════════════════════════════════════════════════════════

class Persona:
    """Clase padre que representa a cualquier persona en el sistema."""

    def __init__(self, nombre: str, correo: str, telefono: str):
        # Atributos privados (encapsulamiento)
        self.__nombre   = nombre
        self.__correo   = correo
        self.__telefono = telefono

    # ── Getters ──────────────────────────────────────────────────────────────
    def get_nombre(self):   return self.__nombre
    def get_correo(self):   return self.__correo
    def get_telefono(self): return self.__telefono

    # ── Setters ──────────────────────────────────────────────────────────────
    def set_nombre(self, v):   self.__nombre   = v
    def set_correo(self, v):   self.__correo   = v
    def set_telefono(self, v): self.__telefono = v

    def descripcion(self):
        return (f"[Persona] {self.__nombre} | "
                f"correo: {self.__correo} | tel: {self.__telefono}")


class Cliente(Persona):
    """Persona natural o jurídica que invierte o ahorra en la plataforma."""

    def __init__(self, nombre: str, correo: str, telefono: str,
                 perfil_riesgo: str, saldo_inicial: float):
        super().__init__(nombre, correo, telefono)
        self.__perfil_riesgo  = perfil_riesgo   # conservador / moderado / agresivo
        self.__saldo_inicial  = saldo_inicial

    def get_perfil_riesgo(self):  return self.__perfil_riesgo
    def get_saldo_inicial(self):  return self.__saldo_inicial
    def set_perfil_riesgo(self, v): self.__perfil_riesgo = v
    def set_saldo_inicial(self, v): self.__saldo_inicial = v

    def descripcion(self):
        base = super().descripcion().replace("[Persona]", "[Cliente]")
        return (f"{base} | perfil: {self.__perfil_riesgo} "
                f"| saldo inicial: ${self.__saldo_inicial:,.0f}")


class Analista(Persona):
    """Profesional interno que gestiona portafolios en la plataforma."""

    def __init__(self, nombre: str, correo: str, telefono: str,
                 especialidad: str, certificacion: str):
        super().__init__(nombre, correo, telefono)
        self.__especialidad  = especialidad
        self.__certificacion = certificacion

    def get_especialidad(self):   return self.__especialidad
    def get_certificacion(self):  return self.__certificacion

    def descripcion(self):
        base = super().descripcion().replace("[Persona]", "[Analista]")
        return f"{base} | esp: {self.__especialidad} | cert: {self.__certificacion}"


# ══════════════════════════════════════════════════════════════════════════════
#  JERARQUÍA 2 · Activo → Inversion / CuentaBancaria
# ══════════════════════════════════════════════════════════════════════════════

class Activo:
    """Clase padre que representa cualquier activo financiero."""

    def __init__(self, nombre: str, valor: float, id_cliente: int):
        self.__nombre     = nombre
        self.__valor      = valor
        self.__id_cliente = id_cliente

    def get_nombre(self):     return self.__nombre
    def get_valor(self):      return self.__valor
    def get_id_cliente(self): return self.__id_cliente
    def set_valor(self, v):   self.__valor = v

    def descripcion(self):
        return (f"[Activo] {self.__nombre} | "
                f"valor: ${self.__valor:,.0f} | cliente_id: {self.__id_cliente}")


class Inversion(Activo):
    """Activo de tipo inversión con tasa de retorno esperada."""

    def __init__(self, nombre: str, valor: float, id_cliente: int,
                 tipo: str, tasa_retorno: float, plazo_meses: int):
        super().__init__(nombre, valor, id_cliente)
        self.__tipo         = tipo          # acciones / bonos / finca raíz / ETF
        self.__tasa_retorno = tasa_retorno  # porcentaje anual
        self.__plazo_meses  = plazo_meses

    def get_tipo(self):         return self.__tipo
    def get_tasa_retorno(self): return self.__tasa_retorno
    def get_plazo_meses(self):  return self.__plazo_meses

    def calcular_rendimiento(self):
        """Retorna el rendimiento estimado al vencimiento del plazo."""
        return self.get_valor() * (self.__tasa_retorno / 100) * (self.__plazo_meses / 12)

    def descripcion(self):
        base = super().descripcion().replace("[Activo]", "[Inversión]")
        rend = self.calcular_rendimiento()
        return (f"{base} | tipo: {self.__tipo} | tasa: {self.__tasa_retorno}% "
                f"| plazo: {self.__plazo_meses} m | rend. est.: ${rend:,.0f}")


class CuentaBancaria(Activo):
    """Activo de tipo cuenta bancaria (ahorros o corriente)."""

    def __init__(self, nombre: str, valor: float, id_cliente: int,
                 banco: str, tipo_cuenta: str, numero_cuenta: str):
        super().__init__(nombre, valor, id_cliente)
        self.__banco          = banco
        self.__tipo_cuenta    = tipo_cuenta    # ahorros / corriente
        self.__numero_cuenta  = numero_cuenta

    def get_banco(self):         return self.__banco
    def get_tipo_cuenta(self):   return self.__tipo_cuenta
    def get_numero_cuenta(self): return self.__numero_cuenta

    def descripcion(self):
        base = super().descripcion().replace("[Activo]", "[Cuenta]")
        return (f"{base} | banco: {self.__banco} | "
                f"tipo: {self.__tipo_cuenta} | nro: {self.__numero_cuenta}")


# ══════════════════════════════════════════════════════════════════════════════
#  JERARQUÍA 3 · Transaccion → Ingreso / Egreso
# ══════════════════════════════════════════════════════════════════════════════

class Transaccion:
    """Clase padre que representa cualquier movimiento financiero."""

    def __init__(self, concepto: str, monto: float,
                 fecha: str, id_cliente: int):
        self.__concepto   = concepto
        self.__monto      = monto
        self.__fecha      = fecha
        self.__id_cliente = id_cliente

    def get_concepto(self):   return self.__concepto
    def get_monto(self):      return self.__monto
    def get_fecha(self):      return self.__fecha
    def get_id_cliente(self): return self.__id_cliente
    def set_monto(self, v):   self.__monto = v

    def descripcion(self):
        return (f"[Transacción] {self.__concepto} | "
                f"${self.__monto:,.0f} | {self.__fecha} | cliente_id: {self.__id_cliente}")


class Ingreso(Transaccion):
    """Flujo positivo de dinero (salario, dividendo, venta de activo…)."""

    def __init__(self, concepto: str, monto: float, fecha: str,
                 id_cliente: int, fuente: str):
        super().__init__(concepto, monto, fecha, id_cliente)
        self.__fuente = fuente   # laboral / inversión / negocio / otro

    def get_fuente(self): return self.__fuente

    def descripcion(self):
        base = super().descripcion().replace("[Transacción]", "[Ingreso ↑]")
        return f"{base} | fuente: {self.__fuente}"


class Egreso(Transaccion):
    """Flujo negativo de dinero (gasto, deuda, impuesto…)."""

    def __init__(self, concepto: str, monto: float, fecha: str,
                 id_cliente: int, categoria: str):
        super().__init__(concepto, monto, fecha, id_cliente)
        self.__categoria = categoria   # vivienda / alimentación / deuda / otro

    def get_categoria(self): return self.__categoria

    def descripcion(self):
        base = super().descripcion().replace("[Transacción]", "[Egreso ↓]")
        return f"{base} | categoría: {self.__categoria}"


print("✅ Todas las clases definidas correctamente.")
print("   Jerarquías: Persona · Activo · Transaccion")

---
## Celda 3 · Capa de Persistencia — Conexión y Creación de Tablas SQLite

In [ ]:
def obtener_conexion():
    """Retorna una conexión activa a la base de datos SQLite."""
    try:
        conn = sqlite3.connect(DB_NAME)
        conn.execute("PRAGMA foreign_keys = ON")  # Activar integridad referencial
        return conn
    except sqlite3.Error as e:
        print(f"❌ Error de conexión a la BD: {e}")
        return None


def crear_tablas():
    """
    Crea las tres tablas relacionales si no existen.

    Relaciones:
      clientes  ←── activos       (activos.id_cliente  → clientes.id)
      clientes  ←── transacciones (transacciones.id_cliente → clientes.id)
    """
    conn = obtener_conexion()
    if conn is None:
        return

    try:
        cursor = conn.cursor()

        # ── Tabla 1: clientes ─────────────────────────────────────────────────
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS clientes (
                id            INTEGER PRIMARY KEY AUTOINCREMENT,
                nombre        TEXT    NOT NULL,
                correo        TEXT    NOT NULL UNIQUE,
                telefono      TEXT,
                perfil_riesgo TEXT    NOT NULL DEFAULT 'moderado',
                saldo_inicial REAL    NOT NULL DEFAULT 0.0
            )
        """)

        # ── Tabla 2: activos ──────────────────────────────────────────────────
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS activos (
                id            INTEGER PRIMARY KEY AUTOINCREMENT,
                nombre        TEXT    NOT NULL,
                tipo          TEXT    NOT NULL,
                valor         REAL    NOT NULL,
                tasa_retorno  REAL    DEFAULT 0.0,
                plazo_meses   INTEGER DEFAULT 0,
                banco         TEXT,
                tipo_cuenta   TEXT,
                id_cliente    INTEGER NOT NULL,
                FOREIGN KEY (id_cliente) REFERENCES clientes(id)
                    ON DELETE CASCADE
            )
        """)

        # ── Tabla 3: transacciones ────────────────────────────────────────────
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS transacciones (
                id          INTEGER PRIMARY KEY AUTOINCREMENT,
                tipo        TEXT    NOT NULL,
                concepto    TEXT    NOT NULL,
                monto       REAL    NOT NULL,
                fecha       TEXT    NOT NULL,
                detalle     TEXT,
                id_cliente  INTEGER NOT NULL,
                FOREIGN KEY (id_cliente) REFERENCES clientes(id)
                    ON DELETE CASCADE
            )
        """)

        conn.commit()
        print("✅ Tablas creadas / verificadas exitosamente:")
        print("   📋 clientes | 💰 activos | 💸 transacciones")

    except sqlite3.Error as e:
        print(f"❌ Error al crear tablas: {e}")
    finally:
        conn.close()


crear_tablas()

---
## Celda 4 · Datos Iniciales de Auditoría (≥5 registros por tabla)

In [ ]:
def insertar_datos_iniciales():
    """
    Inserta datos de muestra para permitir auditoría.
    Solo inserta si las tablas están vacías.
    """
    conn = obtener_conexion()
    if conn is None:
        return

    try:
        cursor = conn.cursor()

        # ── Verificar si ya hay datos ─────────────────────────────────────────
        cursor.execute("SELECT COUNT(*) FROM clientes")
        if cursor.fetchone()[0] > 0:
            print("ℹ️  La base de datos ya contiene registros. No se reinsertan.")
            return

        # ── 6 Clientes ────────────────────────────────────────────────────────
        clientes = [
            ("Valentina Torres",   "v.torres@mail.co",   "3001234567", "moderado",    15_000_000),
            ("Carlos Mendoza",     "c.mendoza@mail.co",  "3109876543", "agresivo",    42_000_000),
            ("Lucía Fernández",    "l.fernandez@mail.co","3157654321", "conservador", 8_500_000),
            ("Andrés Ríos",        "a.rios@mail.co",     "3204567890", "moderado",    25_000_000),
            ("Sofía Castellanos",  "s.castellanos@co",  "3003456789", "agresivo",    60_000_000),
            ("Miguel Herrera",     "m.herrera@mail.co",  "3112345678", "conservador", 5_000_000),
        ]
        cursor.executemany("""
            INSERT INTO clientes (nombre, correo, telefono, perfil_riesgo, saldo_inicial)
            VALUES (?, ?, ?, ?, ?)
        """, clientes)

        # ── 6 Activos ─────────────────────────────────────────────────────────
        activos = [
            # nombre,              tipo,         valor,       tasa,  plazo, banco,       tipo_cta, id_cli
            ("Acciones Bancolombia","inversion",  8_000_000,   12.5,   24,   None,        None,     1),
            ("ETF S&P 500",         "inversion",  18_000_000,  15.0,   36,   None,        None,     2),
            ("CDT Banco de Bogotá", "inversion",  5_000_000,    8.0,   12,   None,        None,     3),
            ("Cuenta Ahorros Nequi","cuenta",     3_500_000,    0.0,    0,  "Nequi",     "ahorros",4),
            ("Fondo de Capital",    "inversion",  30_000_000,  18.0,   48,   None,        None,     5),
            ("Cuenta Corriente",    "cuenta",     1_200_000,    0.0,    0,  "Bancolombia","corriente",6),
        ]
        cursor.executemany("""
            INSERT INTO activos
                (nombre, tipo, valor, tasa_retorno, plazo_meses, banco, tipo_cuenta, id_cliente)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        """, activos)

        # ── 7 Transacciones ───────────────────────────────────────────────────
        hoy = datetime.today().strftime("%Y-%m-%d")
        transacciones = [
            # tipo,      concepto,               monto,       fecha, detalle,          id_cli
            ("ingreso",  "Salario mensual",       4_500_000,   hoy,  "laboral",         1),
            ("egreso",   "Arriendo apartamento",  1_800_000,   hoy,  "vivienda",        1),
            ("ingreso",  "Dividendos acciones",   2_100_000,   hoy,  "inversión",       2),
            ("egreso",   "Cuota préstamo vehículo",900_000,    hoy,  "deuda",           2),
            ("ingreso",  "Venta activo fijo",     12_000_000,  hoy,  "negocio",         5),
            ("egreso",   "Pago impuesto de renta",3_500_000,   hoy,  "impuestos",       5),
            ("ingreso",  "Freelance diseño web",  800_000,     hoy,  "otro",            3),
        ]
        cursor.executemany("""
            INSERT INTO transacciones
                (tipo, concepto, monto, fecha, detalle, id_cliente)
            VALUES (?, ?, ?, ?, ?, ?)
        """, transacciones)

        conn.commit()
        print("✅ Datos de auditoría insertados:")
        print("   👤 6 clientes | 💰 6 activos | 💸 7 transacciones")

    except sqlite3.Error as e:
        print(f"❌ Error al insertar datos: {e}")
    finally:
        conn.close()


insertar_datos_iniciales()

---
## Celda 5 · Funciones CRUD — Clientes

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  CRUD — CLIENTES
# ══════════════════════════════════════════════════════════════════════════════

def crear_cliente(nombre, correo, telefono, perfil_riesgo, saldo_inicial):
    """Inserta un nuevo cliente en la base de datos."""
    conn = obtener_conexion()
    if conn is None: return
    try:
        c = Cliente(nombre, correo, telefono, perfil_riesgo, saldo_inicial)
        conn.execute("""
            INSERT INTO clientes (nombre, correo, telefono, perfil_riesgo, saldo_inicial)
            VALUES (?, ?, ?, ?, ?)
        """, (c.get_nombre(), c.get_correo(), c.get_telefono(),
               c.get_perfil_riesgo(), c.get_saldo_inicial()))
        conn.commit()
        print(f"✅ Cliente '{nombre}' registrado exitosamente.")
    except sqlite3.IntegrityError:
        print(f"⚠️  Ya existe un cliente con el correo '{correo}'.")
    except sqlite3.Error as e:
        print(f"❌ Error al crear cliente: {e}")
    finally:
        conn.close()


def leer_clientes():
    """Muestra todos los clientes registrados."""
    conn = obtener_conexion()
    if conn is None: return
    try:
        cursor = conn.execute("SELECT * FROM clientes ORDER BY id")
        filas  = cursor.fetchall()
        if not filas:
            print("ℹ️  No hay clientes registrados.")
            return
        print(f"\n{'ID':<4} {'Nombre':<22} {'Correo':<28} {'Perfil':<13} {'Saldo Inicial':>14}")
        print("─" * 84)
        for f in filas:
            print(f"{f[0]:<4} {f[1]:<22} {f[2]:<28} {f[4]:<13} ${f[5]:>13,.0f}")
    except sqlite3.Error as e:
        print(f"❌ Error al leer clientes: {e}")
    finally:
        conn.close()


def actualizar_cliente(id_cliente, nuevo_perfil, nuevo_saldo):
    """Actualiza el perfil de riesgo y saldo de un cliente."""
    conn = obtener_conexion()
    if conn is None: return
    try:
        cursor = conn.execute("""
            UPDATE clientes SET perfil_riesgo=?, saldo_inicial=?
            WHERE id=?
        """, (nuevo_perfil, nuevo_saldo, id_cliente))
        conn.commit()
        if cursor.rowcount == 0:
            print(f"⚠️  No se encontró cliente con ID {id_cliente}.")
        else:
            print(f"✅ Cliente ID {id_cliente} actualizado correctamente.")
    except sqlite3.Error as e:
        print(f"❌ Error al actualizar cliente: {e}")
    finally:
        conn.close()


def eliminar_cliente(id_cliente):
    """Elimina un cliente y sus activos/transacciones (CASCADE)."""
    conn = obtener_conexion()
    if conn is None: return
    try:
        cursor = conn.execute("DELETE FROM clientes WHERE id=?", (id_cliente,))
        conn.commit()
        if cursor.rowcount == 0:
            print(f"⚠️  No se encontró cliente con ID {id_cliente}.")
        else:
            print(f"✅ Cliente ID {id_cliente} y sus datos eliminados.")
    except sqlite3.Error as e:
        print(f"❌ Error al eliminar cliente: {e}")
    finally:
        conn.close()


print("✅ Funciones CRUD de Clientes listas.")

---
## Celda 6 · Funciones CRUD — Activos

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  CRUD — ACTIVOS
# ══════════════════════════════════════════════════════════════════════════════

def crear_activo(nombre, tipo, valor, id_cliente,
                 tasa=0.0, plazo=0, banco=None, tipo_cuenta=None):
    """Registra un nuevo activo para un cliente."""
    conn = obtener_conexion()
    if conn is None: return
    try:
        conn.execute("""
            INSERT INTO activos
                (nombre, tipo, valor, tasa_retorno, plazo_meses, banco, tipo_cuenta, id_cliente)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        """, (nombre, tipo, valor, tasa, plazo, banco, tipo_cuenta, id_cliente))
        conn.commit()
        print(f"✅ Activo '{nombre}' registrado para cliente ID {id_cliente}.")
    except sqlite3.IntegrityError:
        print(f"⚠️  El cliente ID {id_cliente} no existe.")
    except sqlite3.Error as e:
        print(f"❌ Error al crear activo: {e}")
    finally:
        conn.close()


def leer_activos():
    """Muestra todos los activos con el nombre de su cliente (JOIN)."""
    conn = obtener_conexion()
    if conn is None: return
    try:
        cursor = conn.execute("""
            SELECT a.id, c.nombre AS cliente, a.nombre, a.tipo,
                   a.valor, a.tasa_retorno, a.plazo_meses
            FROM activos a
            JOIN clientes c ON a.id_cliente = c.id
            ORDER BY a.id
        """)
        filas = cursor.fetchall()
        if not filas:
            print("ℹ️  No hay activos registrados.")
            return
        print(f"\n{'ID':<4} {'Cliente':<20} {'Activo':<22} {'Tipo':<11} "
              f"{'Valor':>14} {'Tasa%':>6} {'Meses':>6}")
        print("─" * 88)
        for f in filas:
            print(f"{f[0]:<4} {f[1]:<20} {f[2]:<22} {f[3]:<11} "
                  f"${f[4]:>13,.0f} {f[5]:>6.1f} {f[6]:>6}")
    except sqlite3.Error as e:
        print(f"❌ Error al leer activos: {e}")
    finally:
        conn.close()


def actualizar_activo(id_activo, nuevo_valor, nueva_tasa):
    """Actualiza el valor y la tasa de retorno de un activo."""
    conn = obtener_conexion()
    if conn is None: return
    try:
        cursor = conn.execute("""
            UPDATE activos SET valor=?, tasa_retorno=? WHERE id=?
        """, (nuevo_valor, nueva_tasa, id_activo))
        conn.commit()
        if cursor.rowcount == 0:
            print(f"⚠️  No se encontró activo con ID {id_activo}.")
        else:
            print(f"✅ Activo ID {id_activo} actualizado.")
    except sqlite3.Error as e:
        print(f"❌ Error al actualizar activo: {e}")
    finally:
        conn.close()


def eliminar_activo(id_activo):
    """Elimina un activo por su ID."""
    conn = obtener_conexion()
    if conn is None: return
    try:
        cursor = conn.execute("DELETE FROM activos WHERE id=?", (id_activo,))
        conn.commit()
        if cursor.rowcount == 0:
            print(f"⚠️  No se encontró activo con ID {id_activo}.")
        else:
            print(f"✅ Activo ID {id_activo} eliminado.")
    except sqlite3.Error as e:
        print(f"❌ Error al eliminar activo: {e}")
    finally:
        conn.close()


print("✅ Funciones CRUD de Activos listas.")

---
## Celda 7 · Funciones CRUD — Transacciones

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  CRUD — TRANSACCIONES
# ══════════════════════════════════════════════════════════════════════════════

def crear_transaccion(tipo, concepto, monto, id_cliente, detalle=""):
    """Registra un ingreso o egreso para un cliente."""
    conn = obtener_conexion()
    if conn is None: return
    try:
        fecha = datetime.today().strftime("%Y-%m-%d")
        conn.execute("""
            INSERT INTO transacciones (tipo, concepto, monto, fecha, detalle, id_cliente)
            VALUES (?, ?, ?, ?, ?, ?)
        """, (tipo, concepto, monto, fecha, detalle, id_cliente))
        conn.commit()
        signo = "↑" if tipo == "ingreso" else "↓"
        print(f"✅ Transacción {signo} '{concepto}' registrada (${monto:,.0f}).")
    except sqlite3.IntegrityError:
        print(f"⚠️  El cliente ID {id_cliente} no existe.")
    except sqlite3.Error as e:
        print(f"❌ Error al crear transacción: {e}")
    finally:
        conn.close()


def leer_transacciones():
    """Muestra todas las transacciones con el nombre del cliente (JOIN)."""
    conn = obtener_conexion()
    if conn is None: return
    try:
        cursor = conn.execute("""
            SELECT t.id, c.nombre AS cliente, t.tipo, t.concepto,
                   t.monto, t.fecha, t.detalle
            FROM transacciones t
            JOIN clientes c ON t.id_cliente = c.id
            ORDER BY t.id
        """)
        filas = cursor.fetchall()
        if not filas:
            print("ℹ️  No hay transacciones registradas.")
            return
        print(f"\n{'ID':<4} {'Cliente':<20} {'Tipo':<9} {'Concepto':<28} "
              f"{'Monto':>13} {'Fecha':<12}")
        print("─" * 92)
        for f in filas:
            signo = "↑" if f[2] == "ingreso" else "↓"
            print(f"{f[0]:<4} {f[1]:<20} {signo+f[2]:<9} {f[3]:<28} "
                  f"${f[4]:>12,.0f} {f[5]:<12}")
    except sqlite3.Error as e:
        print(f"❌ Error al leer transacciones: {e}")
    finally:
        conn.close()


def actualizar_transaccion(id_tx, nuevo_concepto, nuevo_monto):
    """Actualiza el concepto y monto de una transacción."""
    conn = obtener_conexion()
    if conn is None: return
    try:
        cursor = conn.execute("""
            UPDATE transacciones SET concepto=?, monto=? WHERE id=?
        """, (nuevo_concepto, nuevo_monto, id_tx))
        conn.commit()
        if cursor.rowcount == 0:
            print(f"⚠️  No se encontró transacción con ID {id_tx}.")
        else:
            print(f"✅ Transacción ID {id_tx} actualizada.")
    except sqlite3.Error as e:
        print(f"❌ Error al actualizar transacción: {e}")
    finally:
        conn.close()


def eliminar_transaccion(id_tx):
    """Elimina una transacción por su ID."""
    conn = obtener_conexion()
    if conn is None: return
    try:
        cursor = conn.execute("DELETE FROM transacciones WHERE id=?", (id_tx,))
        conn.commit()
        if cursor.rowcount == 0:
            print(f"⚠️  No se encontró transacción con ID {id_tx}.")
        else:
            print(f"✅ Transacción ID {id_tx} eliminada.")
    except sqlite3.Error as e:
        print(f"❌ Error al eliminar transacción: {e}")
    finally:
        conn.close()


print("✅ Funciones CRUD de Transacciones listas.")

---
## Celda 8 · Consulta JOIN — Resumen Financiero por Cliente

In [ ]:
def resumen_financiero_por_cliente():
    """
    Consulta JOIN que une clientes + activos + transacciones.
    Muestra: nombre del cliente, total en activos, total ingresos, total egresos y flujo neto.
    """
    conn = obtener_conexion()
    if conn is None: return
    try:
        cursor = conn.execute("""
            SELECT
                c.id,
                c.nombre,
                c.perfil_riesgo,
                COALESCE(SUM(CASE WHEN a.tipo IS NOT NULL THEN a.valor ELSE 0 END), 0) AS total_activos,
                COALESCE(SUM(CASE WHEN t.tipo = 'ingreso' THEN t.monto ELSE 0 END), 0) AS total_ingresos,
                COALESCE(SUM(CASE WHEN t.tipo = 'egreso'  THEN t.monto ELSE 0 END), 0) AS total_egresos
            FROM clientes c
            LEFT JOIN activos       a ON a.id_cliente = c.id
            LEFT JOIN transacciones t ON t.id_cliente = c.id
            GROUP BY c.id
            ORDER BY c.id
        """)
        filas = cursor.fetchall()

        print("\n" + "═" * 90)
        print("  📊  RESUMEN FINANCIERO POR CLIENTE (JOIN: clientes + activos + transacciones)")
        print("═" * 90)
        print(f"{'ID':<4} {'Nombre':<22} {'Perfil':<13} "
              f"{'Activos':>15} {'Ingresos':>14} {'Egresos':>13} {'Flujo Neto':>14}")
        print("─" * 90)
        for f in filas:
            flujo = f[4] - f[5]
            indicador = "🟢" if flujo >= 0 else "🔴"
            print(f"{f[0]:<4} {f[1]:<22} {f[2]:<13} "
                  f"${f[3]:>14,.0f} ${f[4]:>13,.0f} ${f[5]:>12,.0f} "
                  f"{indicador}${flujo:>12,.0f}")
        print("═" * 90)
    except sqlite3.Error as e:
        print(f"❌ Error en consulta JOIN: {e}")
    finally:
        conn.close()


# Ejecutar el reporte de prueba
resumen_financiero_por_cliente()

---
## Celda 9 · Menú Interactivo Principal (while + try-except)

In [ ]:
def menu_clientes():
    """Submenú CRUD para gestión de clientes."""
    while True:
        print("""
  ┌─ GESTIÓN DE CLIENTES ───────────────────┐
  │  1. Ver todos los clientes              │
  │  2. Registrar nuevo cliente             │
  │  3. Actualizar perfil y saldo           │
  │  4. Eliminar cliente                    │
  │  0. Volver al menú principal            │
  └─────────────────────────────────────────┘""")
        try:
            op = int(input("  Seleccione una opción: "))
        except ValueError:
            print("  ⚠️  Ingrese un número válido.")
            continue

        if op == 1:
            leer_clientes()
        elif op == 2:
            nombre    = input("  Nombre completo: ").strip()
            correo    = input("  Correo electrónico: ").strip()
            telefono  = input("  Teléfono: ").strip()
            print("  Perfil de riesgo: conservador / moderado / agresivo")
            perfil    = input("  Perfil: ").strip().lower()
            try:
                saldo = float(input("  Saldo inicial ($): "))
            except ValueError:
                print("  ⚠️  Saldo inválido. Se usará 0.")
                saldo = 0.0
            crear_cliente(nombre, correo, telefono, perfil, saldo)
        elif op == 3:
            leer_clientes()
            try:
                cid    = int(input("  ID del cliente a actualizar: "))
                perfil = input("  Nuevo perfil (conservador/moderado/agresivo): ").strip()
                saldo  = float(input("  Nuevo saldo inicial: "))
            except ValueError:
                print("  ⚠️  Entrada inválida.")
                continue
            actualizar_cliente(cid, perfil, saldo)
        elif op == 4:
            leer_clientes()
            try:
                cid = int(input("  ID del cliente a eliminar: "))
            except ValueError:
                print("  ⚠️  ID inválido.")
                continue
            confirmar = input(f"  ¿Eliminar cliente ID {cid} y todos sus datos? (s/n): ")
            if confirmar.lower() == "s":
                eliminar_cliente(cid)
        elif op == 0:
            break
        else:
            print("  ⚠️  Opción no válida.")


def menu_activos():
    """Submenú CRUD para gestión de activos financieros."""
    while True:
        print("""
  ┌─ GESTIÓN DE ACTIVOS ────────────────────┐
  │  1. Ver todos los activos (con JOIN)    │
  │  2. Registrar inversión                 │
  │  3. Registrar cuenta bancaria           │
  │  4. Actualizar valor y tasa             │
  │  5. Eliminar activo                     │
  │  0. Volver                              │
  └─────────────────────────────────────────┘""")
        try:
            op = int(input("  Seleccione una opción: "))
        except ValueError:
            print("  ⚠️  Ingrese un número válido.")
            continue

        if op == 1:
            leer_activos()
        elif op == 2:
            leer_clientes()
            try:
                cid   = int(input("  ID del cliente: "))
                nombre= input("  Nombre de la inversión: ").strip()
                tipo  = input("  Tipo (acciones/bonos/ETF/CDT/finca raíz): ").strip()
                valor = float(input("  Valor ($): "))
                tasa  = float(input("  Tasa de retorno anual (%): "))
                plazo = int(input("  Plazo en meses: "))
            except ValueError:
                print("  ⚠️  Dato numérico inválido.")
                continue
            crear_activo(nombre, "inversion", valor, cid, tasa, plazo)
        elif op == 3:
            leer_clientes()
            try:
                cid  = int(input("  ID del cliente: "))
                nombre = input("  Nombre de la cuenta: ").strip()
                banco  = input("  Banco: ").strip()
                tcta   = input("  Tipo (ahorros/corriente): ").strip()
                valor  = float(input("  Saldo actual ($): "))
            except ValueError:
                print("  ⚠️  Dato inválido.")
                continue
            crear_activo(nombre, "cuenta", valor, cid, banco=banco, tipo_cuenta=tcta)
        elif op == 4:
            leer_activos()
            try:
                aid   = int(input("  ID del activo: "))
                valor = float(input("  Nuevo valor: "))
                tasa  = float(input("  Nueva tasa (%): "))
            except ValueError:
                print("  ⚠️  Dato inválido.")
                continue
            actualizar_activo(aid, valor, tasa)
        elif op == 5:
            leer_activos()
            try:
                aid = int(input("  ID del activo a eliminar: "))
            except ValueError:
                print("  ⚠️  ID inválido.")
                continue
            eliminar_activo(aid)
        elif op == 0:
            break
        else:
            print("  ⚠️  Opción no válida.")


def menu_transacciones():
    """Submenú CRUD para gestión de ingresos y egresos."""
    while True:
        print("""
  ┌─ GESTIÓN DE TRANSACCIONES ──────────────┐
  │  1. Ver todas las transacciones (JOIN)  │
  │  2. Registrar ingreso                   │
  │  3. Registrar egreso                    │
  │  4. Actualizar transacción              │
  │  5. Eliminar transacción                │
  │  0. Volver                              │
  └─────────────────────────────────────────┘""")
        try:
            op = int(input("  Seleccione una opción: "))
        except ValueError:
            print("  ⚠️  Ingrese un número válido.")
            continue

        if op == 1:
            leer_transacciones()
        elif op == 2:
            leer_clientes()
            try:
                cid     = int(input("  ID del cliente: "))
                concepto= input("  Concepto del ingreso: ").strip()
                monto   = float(input("  Monto ($): "))
                fuente  = input("  Fuente (laboral/inversión/negocio/otro): ").strip()
            except ValueError:
                print("  ⚠️  Dato inválido.")
                continue
            crear_transaccion("ingreso", concepto, monto, cid, fuente)
        elif op == 3:
            leer_clientes()
            try:
                cid      = int(input("  ID del cliente: "))
                concepto = input("  Concepto del egreso: ").strip()
                monto    = float(input("  Monto ($): "))
                categoria= input("  Categoría (vivienda/alimentación/deuda/impuestos/otro): ").strip()
            except ValueError:
                print("  ⚠️  Dato inválido.")
                continue
            crear_transaccion("egreso", concepto, monto, cid, categoria)
        elif op == 4:
            leer_transacciones()
            try:
                tid      = int(input("  ID de la transacción: "))
                concepto = input("  Nuevo concepto: ").strip()
                monto    = float(input("  Nuevo monto: "))
            except ValueError:
                print("  ⚠️  Dato inválido.")
                continue
            actualizar_transaccion(tid, concepto, monto)
        elif op == 5:
            leer_transacciones()
            try:
                tid = int(input("  ID de la transacción: "))
            except ValueError:
                print("  ⚠️  ID inválido.")
                continue
            eliminar_transaccion(tid)
        elif op == 0:
            break
        else:
            print("  ⚠️  Opción no válida.")


# ══════════════════════════════════════════════════════════════════════════════
#  MENÚ PRINCIPAL
# ══════════════════════════════════════════════════════════════════════════════

def menu_principal():
    """Punto de entrada principal de la aplicación."""
    while True:
        print("""
╔══════════════════════════════════════════════╗
║   🏦  EconoApp — Plataforma Económica v2.0   ║
╠══════════════════════════════════════════════╣
║  1. 👤  Gestión de Clientes                  ║
║  2. 💰  Gestión de Activos Financieros       ║
║  3. 💸  Gestión de Transacciones             ║
║  4. 📊  Resumen Financiero (JOIN)            ║
║  0. 🚪  Salir                                ║
╚══════════════════════════════════════════════╝""")
        try:
            opcion = int(input("  Seleccione una opción: "))
        except ValueError:
            print("  ⚠️  Por favor, ingrese un número.")
            continue

        if   opcion == 1: menu_clientes()
        elif opcion == 2: menu_activos()
        elif opcion == 3: menu_transacciones()
        elif opcion == 4: resumen_financiero_por_cliente()
        elif opcion == 0:
            print("\n  👋 ¡Hasta pronto! Datos guardados en", DB_NAME)
            break
        else:
            print("  ⚠️  Opción no válida.")


# ── Iniciar la aplicación ─────────────────────────────────────────────────────
menu_principal()

---
## Celda 10 · Verificación Final de Auditoría
> Ejecuta esta celda para confirmar que las tablas tienen los registros mínimos requeridos.

In [ ]:
def verificar_auditoria():
    """Verifica que cada tabla tenga al menos 5 registros."""
    conn = obtener_conexion()
    if conn is None: return
    try:
        tablas = ["clientes", "activos", "transacciones"]
        print("\n🔍 VERIFICACIÓN DE AUDITORÍA")
        print("─" * 35)
        for tabla in tablas:
            n = conn.execute(f"SELECT COUNT(*) FROM {tabla}").fetchone()[0]
            estado = "✅" if n >= 5 else "❌ (mínimo 5)"
            print(f"  {tabla:<18}: {n:>3} registros  {estado}")
        print("─" * 35)
    except sqlite3.Error as e:
        print(f"❌ Error: {e}")
    finally:
        conn.close()

verificar_auditoria()